# AuNP CytoViva spectral distributions — reproducible analysis

Parse extensionless `speaks` classification reports, match region counts to group-specific spectral libraries, exclude image files whose region count differs from the unique within-group majority, and export per-image and grouped wavelength distributions and publication-ready plots.

**Input layout** (relative to the notebook's working directory):

```text
data/
  reference_library_peak_wavelengths.csv   # from notebook 01
  speaks/                                  # extensionless CytoViva reports
results/                                   # created automatically
```

If reports are stored elsewhere (e.g. Google Drive), set the environment variable `CYTOVIVA_SPEAKS_DIR` before running the notebook, or change `IMAGE_TXT_DIR` in the first code cell. No folder-picker dialog is used. Run from the repository root so relative paths resolve correctly.

**Dependencies:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `openpyxl`, `ipython`.

**Scientific limitations:** region-to-spectrum matching assumes report row order corresponds exactly to spectral-library numbering. Matching counts alone cannot prove identity. RS/RC files are not silently mapped to an unrelated AuNP library. Pixel-level observations are not independent biological replicates. Letters C/D/E/F are treated as classification subgroups, not assigned experimental times without a validated mapping.


## 1. Load reference library and parse CytoViva reports

Reports can be extensionless files ending in `speaks` or `.txt` reports. The folder is configured without a macOS file-picker.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local configuration. For a public GitHub repository, replace these paths
# with relative paths (e.g. data/reference_library_peak_wavelengths.csv and data/speaks).
# Do not use os.environ.get() with an absolute file path as its argument.
LIBRARY_PEAKS_FILE = Path("reference_library_peak_wavelengths.csv"
)
IMAGE_TXT_DIR = Path(
    "speaks"
)
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Library:", LIBRARY_PEAKS_FILE)
print("Reports:", IMAGE_TXT_DIR)

if not LIBRARY_PEAKS_FILE.is_file():
    raise FileNotFoundError(f"Missing library: {LIBRARY_PEAKS_FILE}. Run notebook 01 first.")
if not IMAGE_TXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Reports folder not found: {IMAGE_TXT_DIR}. "
        "Create data/speaks or set CYTOVIVA_SPEAKS_DIR to your folder path."
    )
print("CytoViva reports:", IMAGE_TXT_DIR.absolute())
print("Outputs:", OUTPUT_DIR.absolute())

library = pd.read_csv(LIBRARY_PEAKS_FILE)
required = {"Spectrum_ID", "Peak_Wavelength_nm"}
if not required.issubset(library.columns):
    raise ValueError(f"Peak library must contain columns {required}. Found: {list(library.columns)}")
if library["Peak_Wavelength_nm"].isna().any():
    raise ValueError("Library contains spectra without detected peaks.")

# Each Spectrum_ID is of the form AuNP+C_Walker_1, etc.
library["Library_Key"] = library["Spectrum_ID"].str.extract(r"^(.*)_\d+$")[0]
library["Library_Position"] = pd.to_numeric(
    library["Spectrum_ID"].str.extract(r"_(\d+)$")[0], errors="coerce"
)
if library["Library_Key"].isna().any() or library["Library_Position"].isna().any():
    raise ValueError("Unexpected Spectrum_ID naming pattern in reference library.")
if library.duplicated(["Library_Key", "Library_Position"]).any():
    raise ValueError("Duplicate spectrum positions within a library group.")
for key, frame in library.groupby("Library_Key"):
    actual = sorted(frame["Library_Position"].astype(int).tolist())
    if actual != list(range(1, len(actual) + 1)):
        raise ValueError(f"Non-contiguous spectrum numbering for {key}: {actual[:10]}...")

library_groups = {
    key: frame.sort_values("Library_Position").copy()
    for key, frame in library.groupby("Library_Key")
}
print("Available library groups:", {k: len(v) for k, v in library_groups.items()})

def library_key_for_group(group):
    # Examples: 'AuNP + C' -> 'AuNP+C_Walker'
    # Preserve RS/RC: do NOT silently map serum groups to naked AuNP libraries.
    return re.sub(r"\s+", "", group) + "_Walker"

# Match Region #1 rows, including exports like 'Region #1 (491; [68] 0.014%'.
# The last bracketed integer is Npts; parenthetical numbers are ignored.
REGION_LINE = re.compile(r"^\s*Region\s*#\s*\d+\b", flags=re.IGNORECASE)
BRACKETED_COUNT = re.compile(r"\[\s*(\d+)\s*\]")
UNCLASSIFIED = re.compile(r"^\s*Unclassified\b", flags=re.IGNORECASE)

def parse_image_classification(path):
    content = path.read_text(encoding="utf-8-sig", errors="replace")
    rows, unclassified = [], None
    for line_number, line in enumerate(content.splitlines(), start=1):
        if UNCLASSIFIED.match(line):
            found = BRACKETED_COUNT.findall(line)
            if found:
                unclassified = int(found[-1])
        elif REGION_LINE.match(line):
            found = BRACKETED_COUNT.findall(line)
            if not found:
                raise ValueError(f"{path.name}:{line_number}: Region row has no [Npts]: {line!r}")
            rows.append({"Region_Position": len(rows) + 1,
                         "Npts": int(found[-1]), "Original_Row": line.strip()})
    if not rows:
        raise ValueError(f"No Region rows found in {path.name}; inspect TXT format.")
    return pd.DataFrame(rows), unclassified

# CytoViva "speaks" reports are usually extensionless text files.
# Exclude .hdr and auxiliary files, including "speaks1" / "speak1" variants.
files = sorted(
    p for p in IMAGE_TXT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in ("", ".txt")
    and re.search(r"\bspeaks\s*$", p.stem, flags=re.IGNORECASE)
)
if len({p.stem for p in files}) != len(files):
    raise ValueError("Duplicate TXT filenames detected across subfolders. Give each image a unique filename.")
print(f"Found {len(files)} CytoViva speaks reports (including subfolders).")
if not files:
    raise FileNotFoundError(f"No extensionless speaks reports found under {IMAGE_TXT_DIR}")

# Define an experimental classification group by the filename prefix.
# Examples: "AuNP + C 1 speaks" -> "AuNP + C";
#           "AuNP RS + D 3 speaks" -> "AuNP RS + D".
# The letter (C/D/E/F) is retained because different image sets can use
# different classification libraries.
def classification_group(path):
    stem = re.sub(r"\s+speaks\s*$", "", path.stem, flags=re.IGNORECASE)
    return re.sub(r"\s+\d+\s*$", "", stem).strip()

records = []
for path in files:
    try:
        regions, unclassified = parse_image_classification(path)
        records.append({
            "path": path, "Image": path.stem,
            "Group": classification_group(path),
            "Region_Rows": len(regions),
            "regions": regions, "Unclassified_Pixels": unclassified,
            "Classified_Pixels": int(regions["Npts"].sum()),
            "Parse_Error": ""
        })
    except (ValueError, UnicodeError, OSError) as exc:
        records.append({
            "path": path, "Image": path.stem,
            "Group": classification_group(path),
            "Region_Rows": np.nan,
            "regions": None, "Unclassified_Pixels": np.nan,
            "Classified_Pixels": np.nan, "Parse_Error": str(exc)
        })

if not records:
    raise FileNotFoundError("No extensionless 'speaks' or '.txt' classification reports found.")

# Determine the most common region count WITHIN EACH classification group.
# A tie is ambiguous: exclude the entire tied group rather than guessing.
from collections import Counter
majorities = {}
for group in sorted({r["Group"] for r in records}):
    counts = Counter(
        int(r["Region_Rows"]) for r in records
        if r["Group"] == group and pd.notna(r["Region_Rows"])
    )
    if not counts:
        majorities[group] = (None, "no_valid_files")
        continue
    highest = max(counts.values())
    modes = [count for count, freq in counts.items() if freq == highest]
    majorities[group] = (
        (modes[0], "unique_mode") if len(modes) == 1
        else (None, "ambiguous_tie")
    )

mapped_images, qc = [], []
for rec in records:
    expected, mode_status = majorities[rec["Group"]]
    count = rec["Region_Rows"]
    reason = ""
    if rec["Parse_Error"]:
        reason = "parse_error"
    elif expected is None:
        reason = mode_status
    elif int(count) != expected:
        reason = "differs_from_group_majority"
    elif library_key_for_group(rec["Group"]) not in library_groups:
        reason = "no_matching_library_group"
    elif int(count) != len(library_groups[library_key_for_group(rec["Group"])]):
        reason = "group_majority_does_not_match_library_group"

    qc.append({
        "Image": rec["Image"], "Group": rec["Group"],
        "Region_Rows": count, "Group_Majority_Regions": expected,
        "Library_Spectra": len(library_groups.get(library_key_for_group(rec["Group"]), [])),
        "Classified_Pixels": rec["Classified_Pixels"],
        "Unclassified_Pixels": rec["Unclassified_Pixels"],
        "Total_Pixels": (
            rec["Classified_Pixels"] + rec["Unclassified_Pixels"]
            if pd.notna(rec["Classified_Pixels"]) and
               pd.notna(rec["Unclassified_Pixels"]) else np.nan
        ),
        "Status": "included" if not reason else "excluded",
        "Exclusion_Reason": reason,
        "Parse_Error": rec["Parse_Error"],
        "File": str(rec["path"])
    })
    if reason:
        continue
    matched = rec["regions"].merge(
        library_groups[library_key_for_group(rec["Group"])][
            ["Library_Position", "Spectrum_ID", "Peak_Wavelength_nm"]
        ],
        left_on="Region_Position", right_on="Library_Position",
        validate="one_to_one"
    )
    matched["Image"] = rec["Image"]
    matched["File"] = str(rec["path"])
    mapped_images.append(matched)

image_qc_all = pd.DataFrame(qc)
image_qc_all.to_csv(OUTPUT_DIR / "image_classification_qc_all_files.csv", index=False)
print("\\nClassification groups and modal region counts:")
print(image_qc_all.groupby("Group", dropna=False).agg(
    Files=("Image", "size"),
    Majority_Regions=("Group_Majority_Regions", "first"),
    Included=("Status", lambda x: int((x == "included").sum()))
).to_string())
print("\\nExcluded files:")
print(image_qc_all.loc[image_qc_all.Status == "excluded",
                       ["Image", "Group", "Region_Rows", "Group_Majority_Regions",
                        "Exclusion_Reason"]].to_string(index=False))

if not mapped_images:
    raise ValueError(
        "No image can be mapped safely to the selected peak library. "
        "Each group needs a matching library subset of the same length. "
        "Review results/image_classification_qc_all_files.csv and choose "
        "the corresponding spectral library for each classification group."
    )
mapped = pd.concat(mapped_images, ignore_index=True)
image_qc = image_qc_all.loc[image_qc_all.Status == "included"].copy()
if image_qc["Image"].duplicated().any():
    raise ValueError("Included reports have duplicate image names; rename them uniquely.")
# Aggregate counts for repeated peak wavelengths, separately for each image.
image_counts = (mapped.groupby(["Image", "Peak_Wavelength_nm"], as_index=False, sort=True)
                .agg(Npts=("Npts", "sum")))
image_counts = image_counts.merge(image_qc[["Image", "Classified_Pixels", "Total_Pixels"]], on="Image", validate="many_to_one")
image_counts["Percent_Classified"] = np.where(
    image_counts["Classified_Pixels"] > 0,
    100 * image_counts["Npts"] / image_counts["Classified_Pixels"], np.nan)
image_counts["Percent_Total_Image"] = np.where(
    image_counts["Total_Pixels"] > 0,
    100 * image_counts["Npts"] / image_counts["Total_Pixels"], np.nan)

mapped.to_csv(OUTPUT_DIR / "image_region_to_library_mapping.csv", index=False)
image_counts.to_csv(OUTPUT_DIR / "image_peak_wavelength_counts.csv", index=False)
image_qc.to_csv(OUTPUT_DIR / "image_classification_qc.csv", index=False)
print(f"Included {len(image_qc)} of {len(files)} files across matched library groups")
print("Total classified pixels:", int(image_qc.Classified_Pixels.sum()))
print("Regions per image:", image_qc[["Image", "Region_Rows", "Classified_Pixels"]].to_string(index=False))
display(image_counts.head(15))

## 2. Group images using their filenames (no metadata file required)

Treatment, letter subgroup, and image number are parsed from each `speaks` filename. The letter is not assumed to represent a time point. Counts are preserved per image and aggregated by treatment and letter subgroup.

In [ ]:
# Extract experimental labels directly from the CytoViva report filenames.
# Examples: "AuNP + C 1 speaks", "AuNP RS + D 3 speaks".
NAME_PATTERN = re.compile(
    r"^(?P<Treatment>AuNP(?:\s+(?:RS|RC))?)\s*\+\s*"
    r"(?P<Subgroup>[A-Za-z])(?:\s+(?P<Image_Number>\d+))?\s+speaks$",
    flags=re.IGNORECASE
)

def parse_image_name(name):
    match = NAME_PATTERN.fullmatch(str(name).strip())
    if match is None:
        raise ValueError(f"Cannot extract experimental labels from filename: {name!r}")
    info = match.groupdict()
    info["Treatment"] = " ".join(info["Treatment"].upper().split())
    info["Subgroup"] = info["Subgroup"].upper()
    info["Image_Number"] = int(info["Image_Number"]) if info["Image_Number"] is not None else None
    return info

labels = pd.DataFrame([
    {"Image": name, **parse_image_name(name)}
    for name in image_counts["Image"].unique()
])

# Preserve the original per-image measurements for replicate-level analysis.
image_labeled = image_counts.merge(labels, on="Image", validate="many_to_one")
image_labeled.to_csv(OUTPUT_DIR / "image_peak_wavelength_counts_labeled.csv", index=False)

# Sum counts only within the same treatment and letter subgroup.
group_counts = (
    image_labeled.groupby(["Treatment", "Subgroup", "Peak_Wavelength_nm"], as_index=False)
    .agg(Npts=("Npts", "sum"))
)
group_counts["Group"] = group_counts["Treatment"] + " + " + group_counts["Subgroup"]
group_counts.to_csv(OUTPUT_DIR / "group_peak_wavelength_counts_long.csv", index=False)

pairs = {}
for group, subset in group_counts.groupby("Group", sort=False):
    pairs[f"{group}__wavelength_nm"] = pd.Series(subset["Peak_Wavelength_nm"].to_numpy())
    pairs[f"{group}__Npts"] = pd.Series(subset["Npts"].to_numpy())
frequency_table = pd.DataFrame(pairs)
frequency_table.to_csv(OUTPUT_DIR / "peak_wavelength_counts_by_group.csv", index=False)
print("Images included:", labels.shape[0])
print("Treatment + subgroup combinations:", group_counts["Group"].nunique())
display(labels.head(15))
display(frequency_table.head())

## 3. Expand pixel frequencies into wavelength observations

Each `Npts` count is expanded into that many pixel-level wavelength observations, preserving the correct treatment/subgroup label. **This can produce large files and plots.** Pixel observations are descriptive, not independent experimental replicates.

In [ ]:
def expand_wavelength_counts(table: pd.DataFrame) -> pd.DataFrame:
    groups = []
    for i in range(0, table.shape[1], 2):
        wavelength_col, count_col = table.columns[i:i+2]
        wavelength = pd.to_numeric(table[wavelength_col], errors="coerce")
        counts = pd.to_numeric(table[count_col], errors="coerce")
        valid = wavelength.notna() & counts.notna()
        values = counts[valid].to_numpy()
        if ((values < 0) | (values != np.floor(values))).any():
            raise ValueError(f"Invalid non-integer or negative counts in: {count_col}")
        expanded = np.repeat(wavelength[valid].to_numpy(), values.astype(int))
        groups.append(pd.DataFrame({"Group": str(wavelength_col).removesuffix("__wavelength_nm"), "Peak_Wavelength_nm": expanded}))
    if not groups:
        return pd.DataFrame(columns=["Group", "Peak_Wavelength_nm"])
    return pd.concat(groups, ignore_index=True)

expanded_long = expand_wavelength_counts(frequency_table)
print(expanded_long.groupby("Group").size().rename("Gold-associated pixel observations"))
display(expanded_long.head())

## 4. Export long and wide tables

The long table is suitable for downstream analysis. A wide table is also exported to reproduce the original Excel/CSV workflow. Unequal group lengths are padded with missing values only; missing values must not be plotted as zero wavelengths.

In [ ]:
expanded_long.to_csv(OUTPUT_DIR / "peak_wavelength_observations_long.csv", index=False)
expanded_wide = pd.DataFrame({
    group: pd.Series(data["Peak_Wavelength_nm"].to_numpy())
    for group, data in expanded_long.groupby("Group", sort=False)
})
expanded_wide.to_csv(OUTPUT_DIR / "peak_wavelength_observations_by_group.csv", index=False)
expanded_wide.to_excel(OUTPUT_DIR / "peak_wavelength_observations_by_group.xlsx", index=False)
print("Exported long CSV, wide CSV, and wide Excel tables.")

## 5. Plot observation-level wavelength distributions

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for group, values in expanded_wide.items():
    clean = values.dropna().to_numpy()
    ax.scatter(np.arange(len(clean)), clean, s=12, alpha=0.65, label=group)
ax.set(title="AuNP-associated peak wavelength observations", xlabel="Observation index within group", ylabel="Maximum-intensity wavelength (nm)")
ax.legend(title="Experimental group", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "peak_wavelength_scatter.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "peak_wavelength_scatter.svg", bbox_inches="tight")
plt.show()

## 6. Violin plot — pixel-weighted peak wavelength distribution

The violin density uses one wavelength observation per classified pixel. `cut=0` prevents density tails extending beyond observed wavelengths. Groups with fewer than two observations or no wavelength variation are displayed as points instead of KDE violins. This is a **descriptive** pixel-level visualization, not a test of biological replicates.

In [ ]:
import seaborn as sns

plot_df = expanded_long.copy()
plot_df["Peak_Wavelength_nm"] = pd.to_numeric(plot_df["Peak_Wavelength_nm"], errors="coerce")
plot_df = plot_df.dropna(subset=["Group", "Peak_Wavelength_nm"])
if plot_df.empty:
    raise ValueError("No peak wavelength observations to plot.")

order = list(plot_df["Group"].drop_duplicates())
variable_groups = [g for g in order if len(plot_df.loc[plot_df.Group == g]) >= 2
                   and plot_df.loc[plot_df.Group == g, "Peak_Wavelength_nm"].nunique() >= 2]
constant_groups = [g for g in order if g not in variable_groups]

fig, ax = plt.subplots(figsize=(max(10, 1.3 * len(order)), 7))
if variable_groups:
    sns.violinplot(
        data=plot_df[plot_df["Group"].isin(variable_groups)],
        x="Group", y="Peak_Wavelength_nm", order=order,
        inner="box", cut=0, color="#82B6D9", linewidth=1,
        ax=ax
    )
# Draw a point for groups where KDE cannot be estimated.
for group in constant_groups:
    vals = plot_df.loc[plot_df.Group == group, "Peak_Wavelength_nm"]
    ax.scatter([order.index(group)], [vals.iloc[0]], color="black", zorder=5, s=30)
ax.set(title="AuNP spectral peak wavelength distribution",
       xlabel="Classification group", ylabel="Peak wavelength (nm)")
ax.tick_params(axis="x", rotation=45)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "violin_peak_wavelength_by_group.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "violin_peak_wavelength_by_group.svg", bbox_inches="tight")
plt.show()
print("Violin plots saved in:", OUTPUT_DIR.absolute())

## Methodological notes

- Frequencies are expanded into one observation per gold-associated pixel count. Pixel-level observations should not automatically be treated as independent biological replicates.
- Group labels and exposure times are inherited from the input file; this notebook does not infer missing metadata.
- The output characterizes spectral peak distributions rather than directly measured AuNP diameters.
- Input data and `results/` are typically excluded from version control when they contain unpublished experimental data; commit the notebook and a short usage guide instead.
